# 04 — Chuẩn bị dữ liệu dự đoán thắng/thua

**Phạm vi:** Phần A (nạp, kiểm tra dữ liệu) và phần B (biểu diễn đặc trưng, baseline) của [kế hoạch dự đoán](../docs/plan/du_doan_thang_thua_plan.md). Đây là dự đoán *một ván sau khi chọn đủ 10 tướng*, không phải dự đoán cả series hoặc lúc đang cấm/chọn.

Notebook chỉ **gọi** `liqily.prediction.data` và `liqily.prediction.features`; logic kiểm tra/mã hóa được duy trì trong source Python. Chạy các cell theo thứ tự từ trên xuống. Có thể khởi chạy Jupyter từ thư mục gốc hoặc thư mục `notebook/` với môi trường đã cài package (`python -m pip install -e '.[dev]'`).

**Đầu vào hợp lệ:** `team_a_heroes`, `team_b_heroes`, `team_a_side`, `patch_id`. **Nhãn:** `team_a_win`. Các ID, thời gian, nhãn và thống kê toàn giải không được đưa vào ma trận đặc trưng.

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

from IPython.display import Markdown, display

# Support launching Jupyter from the project root or the notebook directory.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src' / 'liqily').is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError('Run this notebook from the project root or a subdirectory')
sys.path.insert(0, str(project_root / 'src'))

from liqily.prediction.data import DEFAULT_DATASET, audit_dataset, load_hero_names
from liqily.prediction.features import FeatureEncoder, MajorityBaseline, SideBaseline, labels

dataset_dir = DEFAULT_DATASET
schema = json.loads((dataset_dir / 'feature_schema.json').read_text(encoding='utf-8'))
manifest = json.loads((dataset_dir / 'split_manifest.json').read_text(encoding='utf-8'))
assert schema['dataset_version'] == manifest['dataset_version']
assert schema['features'] == ['team_a_heroes', 'team_b_heroes', 'team_a_side', 'patch_id']
assert schema['label'] == 'team_a_win'
print('Dataset version:', schema['dataset_version'])
print('Dataset directory:', dataset_dir)

Dataset version: 035ce761e53a47dbfd18ca1d9fabe7ceb0ec79115271bccac303bed2a6c0195c
Dataset directory: /home/andev/dev/projects/pka/data_mining/proj/dataset/035ce761e53a47db


## A. Nạp và kiểm tra dữ liệu

`audit_dataset()` đọc ba CSV dự đoán, parse JSON của hai đội hình, kiểm tra đủ 5 ID mỗi đội, tướng trùng, phe, patch, nhãn boolean, danh mục tướng và ID/series giao nhau giữa các tập. Hàm đối chiếu số đếm và phân bổ series với `split_manifest.json`. **Không tự ý sửa hoặc xóa dòng lỗi.**

Bộ dữ liệu đã chia theo lịch series; toàn bộ ván của một series phải ở cùng một tập. Lịch này chỉ là đại diện cho thời gian thi đấu, chưa xác minh thời gian thực tế từng ván.

In [2]:
games_by_split, issues = audit_dataset(dataset_dir)
if issues:
    for issue in issues[:20]:
        print(f'[ERROR] {issue.split} | {issue.game_id} | {issue.field}: {issue.message}')
    raise ValueError(f'Dataset audit failed with {len(issues)} issue(s)')

rows = []
for split in ('train', 'validation', 'test'):
    games = games_by_split[split]
    wins = sum(game.team_a_win for game in games)
    actual = {'series': len({game.series_id for game in games}), 'games': len(games),
              'labels': {'false': len(games) - wins, 'true': wins}}
    assert actual == manifest['split_counts'][split]
    rows.append(f'| {split} | {actual["series"]} | {actual["games"]} | {wins} | {len(games) - wins} |')
display(Markdown('| Split | Series | Games | Team A wins | Team A losses |\n'
                 '| --- | ---: | ---: | ---: | ---: |\n' + '\n'.join(rows)))
print('Audit issues:', len(issues))

| Split | Series | Games | Team A wins | Team A losses |
| --- | ---: | ---: | ---: | ---: |
| train | 60 | 231 | 128 | 103 |
| validation | 7 | 28 | 16 | 12 |
| test | 8 | 40 | 27 | 13 |

Audit issues: 0


**Nhận xét phần A:** Snapshot gồm 75 series/299 ván (train 231, validation 28, test 40). Nhãn thắng của đội A lần lượt là 128, 16 và 27. Nếu cell trên báo lỗi, phải điều tra snapshot và quy tắc kiểm tra trước khi đi tiếp; `Audit issues: 0` chỉ có nghĩa các kiểm tra hiện có đã đạt, **không chứng minh khả năng dự đoán**.

`heroes.csv` chỉ giúp kiểm tra/hiển thị tên tướng. Danh mục mới so với train là một tình huống cần xử lý ở phần B, không nhất thiết là lỗi dữ liệu.

## B. Biểu diễn đặc trưng

Chỉ `fit` encoder trên **train** để cố định tập ID tướng và patch. Một vector gồm `H` cột nhị phân tướng đội A, số tướng mới đội A, `H` cột tướng đội B, số tướng mới đội B, một cột `team_a_is_blue`, `P` cột patch và một cột `patch_is_unknown`: tổng cộng **`2H + P + 4`** cột. `frozenset` bảo đảm vị trí trong JSON không bị hiểu nhầm là vai trò hay thứ tự chọn.

Validation/test chỉ dùng `transform`, tuyệt đối không `fit` lại. `team_a_win` được tách riêng thành `y`; `game_id`, `series_id`, thời gian và `split` chỉ là metadata kiểm tra, không là đặc trưng.

In [6]:
train = games_by_split['train']
validation = games_by_split['validation']
test = games_by_split['test']

encoder = FeatureEncoder.fit(train)
X = {split: encoder.transform(games) for split, games in games_by_split.items()}
y = {split: labels(games) for split, games in games_by_split.items()}
width = len(encoder.feature_names)

assert width == 2 * len(encoder.hero_ids) + len(encoder.patch_ids) + 4
assert all(len(vector) == width for matrix in X.values() for vector in matrix)
assert all(len(X[split]) == len(y[split]) for split in X)
assert not {'team_a_win', 'game_id', 'series_id', 'split', 'split_time'} & set(encoder.feature_names)

display(Markdown('| Split | Feature matrix | Labels |\n| --- | ---: | ---: |\n' +
                 '\n'.join(f'| {split} | {len(X[split])} × {width} | {len(y[split])} |'
                           for split in ('train', 'validation', 'test'))))

print('Training hero categories:', len(encoder.hero_ids))
print('Training patch categories:', len(encoder.patch_ids))
print('Training patches:', encoder.patch_ids)
print('Feature count:', width)

| Split | Feature matrix | Labels |
| --- | ---: | ---: |
| train | 231 × 173 | 231 |
| validation | 28 × 173 | 28 |
| test | 40 × 173 | 40 |

Training hero categories: 83
Training patch categories: 3
Training patches: ('aov:vn:2026-02-14', 'aov:vn:2026-03-11', 'aov:vn:2026-04-08')
Feature count: 173


### Minh họa một vector

Các cột nhị phân bằng 1 cho biết tướng thuộc đội A/B hoặc patch của ván. Tên tướng bên dưới chỉ dùng để đọc kết quả; vector vẫn dựa trên ID. Không in toàn bộ 173 cột để giữ notebook dễ đọc.

In [7]:
hero_names = load_hero_names(dataset_dir / 'csv' / 'heroes.csv')
example = train[0]
encoded = dict(zip(encoder.feature_names, encoder.transform_one(example), strict=True))
display(Markdown(f'**Game:** `{example.game_id}`  \n'
                 f'**Team A:** {", ".join(sorted(hero_names[hero] for hero in example.team_a_heroes))}  \n'
                 f'**Team B:** {", ".join(sorted(hero_names[hero] for hero in example.team_b_heroes))}  \n'
                 f'**Side/patch:** `{example.team_a_side}` / `{example.patch_id}`'))
active_rows = []
for feature, value in encoded.items():
    if not value:
        continue
    if feature.startswith(('team_a_hero=', 'team_b_hero=')):
        meaning = hero_names[feature.split('=', 1)[1]]
    elif feature.startswith('patch='):
        meaning = example.patch_id
    else:
        meaning = example.team_a_side
    active_rows.append(f'| `{feature}` | {meaning} | {value} |')
display(Markdown('| Active feature | Meaning | Value |\n'
                 '| --- | --- | ---: |\n' + '\n'.join(active_rows)))
print('Label (separate from features):', int(example.team_a_win))

**Game:** `liquipedia:honorofkings:32206:AOG26S1R01:M1:map1`  
**Team A:** Biron, Hayate, Iggy, Tachi, Toro  
**Team B:** Capheny, Heino, Keera, Marja, Y'bneth  
**Side/patch:** `red` / `aov:vn:2026-02-14`

| Active feature | Meaning | Value |
| --- | --- | ---: |
| `team_a_hero=033fc0fc-5f17-527d-bbf1-7b015c35625c` | Hayate | 1 |
| `team_a_hero=455bea3b-0709-50b7-99ab-6d36a5903fa1` | Iggy | 1 |
| `team_a_hero=9f1f2f8f-b7a7-501b-a521-0b963af37efa` | Tachi | 1 |
| `team_a_hero=b0f937b3-ca67-57ed-90e2-ffdb6f63b876` | Toro | 1 |
| `team_a_hero=eee25327-6ee4-5308-be2f-ea5197a6104d` | Biron | 1 |
| `team_b_hero=46dbb84d-ecf1-58de-887b-a359cb95ee60` | Y'bneth | 1 |
| `team_b_hero=4a963a79-df38-5144-b61f-503f5daa3fb6` | Keera | 1 |
| `team_b_hero=62b51229-8561-5c25-9304-c99e140edb7c` | Marja | 1 |
| `team_b_hero=b66f36ce-d88d-56c2-879a-097f6cba7924` | Heino | 1 |
| `team_b_hero=f6a7ee98-4b62-55b4-aa80-57ce5d792bf2` | Capheny | 1 |
| `patch=aov:vn:2026-02-14` | aov:vn:2026-02-14 | 1 |

Label (separate from features): 1


### Danh mục chưa có trong train

Bảng dưới đây **chỉ mô tả đầu vào** của từng tập; không dùng nhãn validation/test để điều chỉnh encoder. Tướng/patch mới vẫn giữ nguyên số cột nhờ các cột đếm/cờ unknown. Những cột này luôn bằng 0 ở train nên không thể suy diễn rằng mô hình đã học được ảnh hưởng của danh mục mới.

In [8]:
unknown_rows = []
for split, games in games_by_split.items():
    diagnostics = [encoder.unknown_categories(game) for game in games]
    unseen_heroes = {hero for item in diagnostics
                     for hero in item['team_a_heroes'] + item['team_b_heroes']}
    new_hero_games = sum(bool(item['team_a_heroes'] or item['team_b_heroes'])
                         for item in diagnostics)
    new_patch_games = sum(item['patch_id'] is not None for item in diagnostics)
    unknown_rows.append(f'| {split} | {len(unseen_heroes)} | {new_hero_games} | {new_patch_games} |')
display(Markdown('| Split | Unseen hero IDs | Games with unseen heroes | Games with unseen patch |\n'
                 '| --- | ---: | ---: | ---: |\n' + '\n'.join(unknown_rows)))

| Split | Unseen hero IDs | Games with unseen heroes | Games with unseen patch |
| --- | ---: | ---: | ---: |
| train | 0 | 0 | 0 |
| validation | 1 | 3 | 0 |
| test | 2 | 4 | 6 |

**Nhận xét:** Train không có danh mục mới theo định nghĩa. Validation có một ID tướng mới; test có hai ID tướng mới và patch `aov:vn:2026-04-23` chỉ xuất hiện ở test. Các số liệu này dùng để mô tả giới hạn của encoder, không để fit lại hoặc chọn mô hình theo test.

## Baseline xác suất từ train

`MajorityBaseline` trả cùng một xác suất cho mọi ván: tỷ lệ đội A thắng trong train. `SideBaseline` tính tỷ lệ riêng cho phe xanh/đỏ; nếu một phe vắng trong train, dùng tỷ lệ chung. Hai baseline **không dùng đội hình tướng**, và chỉ sử dụng nhãn train khi `fit`. Đây là mốc để so sánh với mô hình ở phần C/D, chưa phải mô hình cuối cùng.

In [9]:
majority = MajorityBaseline.fit(train)
side = SideBaseline.fit(train)
train_by_side = Counter(game.team_a_side for game in train)
assert majority.probability == sum(y['train']) / len(train)
assert side.prior == majority.probability
display(Markdown('| Baseline | Training games | Estimated P(Team A wins) |\n'
                 '| --- | ---: | ---: |\n'
                 f'| Overall | {len(train)} | {majority.probability:.3f} |\n'
                 f'| Team A blue | {train_by_side["blue"]} | {side.blue_probability:.3f} |\n'
                 f'| Team A red | {train_by_side["red"]} | {side.red_probability:.3f} |'))
print('First three validation predictions (overall baseline):',
      majority.predict_proba(validation)[:3])
print('First three validation predictions (side baseline):',
      side.predict_proba(validation)[:3])

| Baseline | Training games | Estimated P(Team A wins) |
| --- | ---: | ---: |
| Overall | 231 | 0.554 |
| Team A blue | 108 | 0.583 |
| Team A red | 123 | 0.528 |

First three validation predictions (overall baseline): [0.5541125541125541, 0.5541125541125541, 0.5541125541125541]
First three validation predictions (side baseline): [0.5833333333333334, 0.5833333333333334, 0.5833333333333334]


## Tổng kết và giới hạn

- Phần A: dữ liệu vượt qua các quy tắc kiểm tra hiện có; mỗi series ở đúng split và số đếm khớp manifest. Việc chia theo lịch series **không bảo đảm** trình tự thời gian thực tế từng ván.
- Phần B: encoder fit trên 231 ván train, ba ma trận cùng 173 cột (83 tướng và 3 patch trong train); nhãn tách khỏi đặc trưng. Danh mục mới ở holdout được ghi nhận, nhưng chưa có tác động học được trong train.
- Baseline chung là `128/231 ≈ 0,554`; baseline theo phe chỉ là tỷ lệ quan sát trong train, không phải bằng chứng quan hệ nhân quả. Với một giải đấu nhỏ, không suy rộng kết quả ra xếp hạng hoặc patch khác.
- **Bước kế tiếp:** huấn luyện và so sánh mô hình trên train/validation ở phần C; chỉ dùng test sau khi đã chốt phương án ở phần D. Notebook này không fit mô hình hoặc đánh giá/chọn mô hình theo test.